# 04a. Línea Base Dummy

**Objetivo:** fijar el piso mínimo de desempeño para validar que el problema es aprendible.
**Entradas (inputs):** `data/splits/train_denoised.csv` y `data/splits/<split>_denoised.csv`.
**Salidas (outputs):** `data/dummy_eval.csv` y `data/dummy_predicciones_<split>.csv`.
**Notebook anterior:** `notebooks/pipeline/03_denoising_reglas_core.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`.

> **Contrato de comparación:** este baseline se evalúa sobre el mismo subconjunto denoised del híbrido (`BASELINE_EVAL_ON=dev` por defecto).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.dummy import DummyClassifier

sys.path.insert(0, str(Path.cwd().resolve()))
from utils_shared import setup_paths, load_splits, calculate_metrics, get_cv_splitter

paths = setup_paths()
DATA_PATH   = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']

EVAL_ON = os.getenv('BASELINE_EVAL_ON', 'dev').strip().lower()
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"BASELINE_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")

print("Entorno de evaluación inicializado.")
print("EVAL_ON:", EVAL_ON)


## 1. Cargar particiones denoised (alineadas con el híbrido)
Se usa `train_denoised.csv` para entrenamiento y `<split>_denoised.csv` para evaluación.
Así, las líneas base y el modelo híbrido comparten exactamente el mismo universo de `row_id`.


In [ ]:
# Cargar particiones denoised para mantener comparabilidad metodológica
train_path = SPLITS_PATH / 'train_denoised.csv'
eval_path = SPLITS_PATH / f'{EVAL_ON}_denoised.csv'

if not train_path.exists() or not eval_path.exists():
    raise FileNotFoundError(
        f'Faltan archivos denoised para baseline. Esperados: {train_path} y {eval_path}. Ejecuta 03 primero.'
    )

df_train = pd.read_csv(train_path)
df_eval = pd.read_csv(eval_path)

X_train, y_train = df_train['texto'], df_train['etiqueta']
X_eval, y_eval = df_eval['texto'], df_eval['etiqueta']

print('Resumen de entrada:')
print(f"  - Entrenamiento (denoised): {len(X_train)} registros")
print(f"  - Evaluación ({EVAL_ON}_denoised): {len(X_eval)} registros")


## 2. Prueba Ciega 1: El Diagnóstico Mayoritario
Si este robot diagnostica a absolutamente todos como 'Depresión', ¿con qué porcentaje de pacientes acierta solo por estadística poblacional?


In [ ]:
# Modalidad 'prior': predice siempre la clase con mayor frecuencia en entrenamiento.
clf_majority = DummyClassifier(strategy='prior')
clf_majority.fit(X_train, y_train)

preds_maj = clf_majority.predict(X_eval)

print(f"--- Resultados en {EVAL_ON} (estrategia prior) ---")
maj_metrics = calculate_metrics(y_eval, preds_maj)
print(maj_metrics['report'])


## 3. Prueba Ciega 2: Lotería Estratificada
Este robot tira una moneda para decidir, pero sabe que la moneda debe caer en 'Depresión' un ~70% de las veces según la demografía del hospital.


In [ ]:
# Modalidad 'stratified': genera predicciones aleatorias respetando la distribución de clases de entrenamiento.
clf_strat = DummyClassifier(strategy='stratified', random_state=42)
clf_strat.fit(X_train, y_train)

preds_strat = clf_strat.predict(X_eval)

print(f"--- Resultados en {EVAL_ON} (estrategia stratified) ---")
strat_metrics = calculate_metrics(y_eval, preds_strat)
print(strat_metrics['report'])


In [ ]:
# Exportar resultados estandarizados (estrategia stratified)
import pandas as pd

eval_df = pd.DataFrame([{
    'modelo': 'dummy',
    'f1_macro': strat_metrics['f1_macro'],
    'precision_macro': strat_metrics['precision_macro'],
    'recall_macro': strat_metrics['recall_macro'],
    'accuracy': strat_metrics['accuracy'],
    'n_train': len(X_train),
    'n_eval': len(X_eval),
    'n_dev': len(X_eval),  # compatibilidad hacia atrás
    'eval_split': EVAL_ON,
}])
output_path = DATA_PATH / 'dummy_eval.csv'
eval_df.to_csv(output_path, index=False)
print(f"Archivo exportado: {output_path}")

pred_df = pd.DataFrame({
    'row_id': df_eval['row_id'].astype(int),
    'y_true': y_eval.astype(str),
    'y_pred': pd.Series(preds_strat).astype(str),
})
pred_path = DATA_PATH / f'dummy_predicciones_{EVAL_ON}.csv'
pred_df.to_csv(pred_path, index=False)
print(f"Archivo exportado: {pred_path}")
